In [32]:
# 1) Charger et valider le JSON d'entree
from __future__ import annotations

import copy
import json
import math
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd

# Constantes operationnelles
HEURE_DEPART_MIN = 7 * 60 + 30
HEURE_RETOUR_LIMITE_MIN = 17 * 60
VITESSE_URBAINE_KMH = 25.0
TEMPS_LEVEE_MIN = 3
TEMPS_VIDANGE_MIN = 20


@dataclass
class Point:
    lat: float
    lng: float


@dataclass
class Camion:
    id: str
    capacite_max_m3: float
    capacite_disponible_m3: float
    position_depart: Point


@dataclass
class Poubelle:
    id: str
    position: Point
    taux_remplissage_pct: float
    volume_nominal_m3: float
    volume_actuel_m3: float


def _assert_keys(obj: Dict[str, Any], keys: List[str], parent: str) -> None:
    for k in keys:
        if k not in obj:
            raise ValueError(f"Champ manquant: {parent}.{k}")


def _charger_payload_depuis_fichiers(base_dir: Path, date_cible: str | None = None) -> Dict[str, Any] | None:
    bins_path = base_dir / "bins.csv"
    daily_path = base_dir / "daily_sensor_readings.csv"
    trucks_path = base_dir / "trucks.csv"
    training_pairs_path = base_dir / "training_pairs.json"
    optimal_routes_path = base_dir / "optimal_routes.json"

    if not (bins_path.exists() and daily_path.exists() and trucks_path.exists()):
        return None

    bins_df = pd.read_csv(bins_path)
    daily_df = pd.read_csv(daily_path)
    trucks_df = pd.read_csv(trucks_path)

    required_bins = {"bin_id", "lat", "lng", "volume_nominal_m3", "zone"}
    required_daily = {"date", "reading_time", "bin_id", "fill_pct", "volume_actual_m3", "sensor_status", "lat", "lng"}
    required_trucks = {"truck_id", "capacity_max_m3", "depot_lat", "depot_lng"}

    if not required_bins.issubset(set(bins_df.columns)):
        raise ValueError("bins.csv ne contient pas les colonnes minimales attendues.")
    if not required_daily.issubset(set(daily_df.columns)):
        raise ValueError("daily_sensor_readings.csv ne contient pas les colonnes minimales attendues.")
    if not required_trucks.issubset(set(trucks_df.columns)):
        raise ValueError("trucks.csv ne contient pas les colonnes minimales attendues.")

    if date_cible is None:
        date_cible = str(daily_df["date"].max())

    daily_date = daily_df[daily_df["date"] == date_cible].copy()
    if daily_date.empty:
        raise ValueError(f"Aucune lecture capteur pour la date {date_cible}.")

    # Prendre la lecture 07:00 si disponible sinon la plus recente de la journee
    if (daily_date["reading_time"] == "07:00:00").any():
        daily_date = daily_date[daily_date["reading_time"] == "07:00:00"].copy()
    else:
        daily_date = daily_date.sort_values(["bin_id", "reading_time"]).groupby("bin_id", as_index=False).tail(1)

    merged = bins_df.merge(
        daily_date[["bin_id", "fill_pct", "volume_actual_m3", "sensor_status", "lat", "lng"]],
        on="bin_id",
        how="left",
        suffixes=("_bin", "_sensor"),
    )

    merged["lat"] = merged["lat_sensor"].fillna(merged["lat_bin"])
    merged["lng"] = merged["lng_sensor"].fillna(merged["lng_bin"])
    merged["fill_pct"] = merged["fill_pct"].fillna(0.0).clip(0, 100)
    merged["volume_actual_m3"] = merged["volume_actual_m3"].fillna(
        merged["volume_nominal_m3"] * merged["fill_pct"] / 100.0
    )
    merged["volume_actual_m3"] = merged[["volume_actual_m3", "volume_nominal_m3"]].min(axis=1).clip(lower=0.0)

    # Toutes les poubelles sont collectees, meme si capteur KO ou volume 0
    poubelles = []
    for _, row in merged.iterrows():
        poubelles.append(
            {
                "id": str(row["bin_id"]),
                "position": {"lat": float(row["lat"]), "lng": float(row["lng"])},
                "taux_remplissage_pct": float(row["fill_pct"]),
                "volume_nominal_m3": float(row["volume_nominal_m3"]),
                "volume_actuel_m3": float(row["volume_actual_m3"]),
            }
        )

    depot_lat = float(trucks_df["depot_lat"].iloc[0])
    depot_lng = float(trucks_df["depot_lng"].iloc[0])

    camions = []
    for _, row in trucks_df.iterrows():
        camions.append(
            {
                "id": str(row["truck_id"]),
                "capacite_max_m3": float(row["capacity_max_m3"]),
                "capacite_disponible_m3": float(row["capacity_max_m3"]),
                "position_depart": {"lat": depot_lat, "lng": depot_lng},
            }
        )

    decharge = {"lat": 36.92, "lng": 10.08}
    if optimal_routes_path.exists():
        with open(optimal_routes_path, "r", encoding="utf-8") as f:
            hist = json.load(f)
            if isinstance(hist, list) and hist:
                landfill = hist[0].get("landfill")
                if isinstance(landfill, dict) and {"lat", "lng"}.issubset(landfill.keys()):
                    decharge = {"lat": float(landfill["lat"]), "lng": float(landfill["lng"])}

    if training_pairs_path.exists():
        with open(training_pairs_path, "r", encoding="utf-8") as f:
            pairs = json.load(f)
            if isinstance(pairs, list) and pairs:
                decharge_pair = pairs[0].get("input", {}).get("decharge")
                if isinstance(decharge_pair, dict) and {"lat", "lng"}.issubset(decharge_pair.keys()):
                    decharge = {"lat": float(decharge_pair["lat"]), "lng": float(decharge_pair["lng"])}

    return {
        "date": date_cible,
        "heure_generation": "07:00",
        "camions": camions,
        "poubelles": poubelles,
        "decharge": decharge,
    }


def charger_payload(path_json: str | None = None, date_cible: str | None = None) -> Dict[str, Any]:
    if path_json is not None:
        with open(path_json, "r", encoding="utf-8") as f:
            return json.load(f)

    base_dir = Path.cwd()
    payload_reel = _charger_payload_depuis_fichiers(base_dir=base_dir, date_cible=date_cible)
    if payload_reel is not None:
        return payload_reel

    # Fallback synthetique si les fichiers reels ne sont pas disponibles
    payload = {
        "date": datetime.now().strftime("%Y-%m-%d"),
        "heure_generation": "07:00",
        "camions": [
            {
                "id": "CAM-01",
                "capacite_max_m3": 12.0,
                "capacite_disponible_m3": 12.0,
                "position_depart": {"lat": 36.7538, "lng": 3.0588},
            },
            {
                "id": "CAM-02",
                "capacite_max_m3": 9.0,
                "capacite_disponible_m3": 9.0,
                "position_depart": {"lat": 36.7538, "lng": 3.0588},
            },
            {
                "id": "CAM-03",
                "capacite_max_m3": 14.0,
                "capacite_disponible_m3": 14.0,
                "position_depart": {"lat": 36.7538, "lng": 3.0588},
            },
        ],
        "poubelles": [],
        "decharge": {"lat": 36.7000, "lng": 3.2000},
    }

    rng = np.random.default_rng(42)
    for i in range(45):
        lat = 36.68 + rng.random() * 0.18
        lng = 2.95 + rng.random() * 0.20
        nominal = float(rng.choice([0.18, 0.24, 0.30]))
        fill = float(rng.uniform(0.0, 1.0))
        payload["poubelles"].append(
            {
                "id": f"PB-{i+1:03d}",
                "position": {"lat": float(lat), "lng": float(lng)},
                "taux_remplissage_pct": round(fill * 100, 2),
                "volume_nominal_m3": nominal,
                "volume_actuel_m3": round(nominal * fill, 4),
            }
        )
    return payload


def valider_et_normaliser_payload(payload: Dict[str, Any]) -> Tuple[str, str, List[Camion], List[Poubelle], Point]:
    _assert_keys(payload, ["date", "heure_generation", "camions", "poubelles", "decharge"], "payload")

    date_str = payload["date"]
    heure_generation = payload["heure_generation"]

    camions: List[Camion] = []
    for c in payload["camions"]:
        _assert_keys(c, ["id", "capacite_max_m3", "capacite_disponible_m3", "position_depart"], "camion")
        _assert_keys(c["position_depart"], ["lat", "lng"], "camion.position_depart")
        camions.append(
            Camion(
                id=str(c["id"]),
                capacite_max_m3=float(c["capacite_max_m3"]),
                capacite_disponible_m3=float(c["capacite_disponible_m3"]),
                position_depart=Point(float(c["position_depart"]["lat"]), float(c["position_depart"]["lng"])),
            )
        )

    poubelles: List[Poubelle] = []
    for p in payload["poubelles"]:
        _assert_keys(p, ["id", "position", "taux_remplissage_pct", "volume_nominal_m3", "volume_actuel_m3"], "poubelle")
        _assert_keys(p["position"], ["lat", "lng"], "poubelle.position")
        volume_nominal = float(p["volume_nominal_m3"])
        volume_actuel = float(p["volume_actuel_m3"])
        if volume_actuel < 0 or volume_actuel > volume_nominal:
            volume_actuel = max(0.0, min(volume_actuel, volume_nominal))

        poubelles.append(
            Poubelle(
                id=str(p["id"]),
                position=Point(float(p["position"]["lat"]), float(p["position"]["lng"])),
                taux_remplissage_pct=float(p["taux_remplissage_pct"]),
                volume_nominal_m3=volume_nominal,
                volume_actuel_m3=volume_actuel,
            )
        )

    _assert_keys(payload["decharge"], ["lat", "lng"], "decharge")
    decharge = Point(float(payload["decharge"]["lat"]), float(payload["decharge"]["lng"]))

    if len(camions) == 0 or len(poubelles) == 0:
        raise ValueError("Le payload doit contenir au moins 1 camion et 1 poubelle.")

    return date_str, heure_generation, camions, poubelles, decharge


payload_jour = charger_payload(path_json=None, date_cible=None)
date_plan, heure_generation, camions, poubelles, decharge = valider_et_normaliser_payload(payload_jour)
print(f"Payload charge: {len(camions)} camions, {len(poubelles)} poubelles | date={date_plan}")

Payload charge: 6 camions, 180 poubelles | date=2024-02-29


In [33]:
# 2) Construire les objets geographiques et la matrice des distances

def haversine_km(a: Point, b: Point) -> float:
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [a.lat, a.lng, b.lat, b.lng])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * r * math.asin(math.sqrt(h))


def construire_noeuds(camions: List[Camion], poubelles: List[Poubelle], decharge: Point) -> Dict[str, Any]:
    depot = camions[0].position_depart
    nodes = [depot] + [p.position for p in poubelles] + [decharge]
    ids = ["DEPOT"] + [p.id for p in poubelles] + ["DECHARGE"]
    idx_by_id = {node_id: i for i, node_id in enumerate(ids)}
    return {"nodes": nodes, "ids": ids, "idx_by_id": idx_by_id, "depot": depot, "decharge": decharge}


def matrice_distances_km(nodes: List[Point]) -> np.ndarray:
    n = len(nodes)
    d = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            dist = haversine_km(nodes[i], nodes[j])
            d[i, j] = dist
            d[j, i] = dist
    return d


geo = construire_noeuds(camions, poubelles, decharge)
D = matrice_distances_km(geo["nodes"])
IDX_DECHARGE = geo["idx_by_id"]["DECHARGE"]
print(f"Matrice distances calculee: {D.shape} | idx_decharge={IDX_DECHARGE}")

Matrice distances calculee: (182, 182) | idx_decharge=181


In [34]:
# 3) Verifier les contraintes de couverture totale et de capacite

def verifier_contraintes_globales(camions: List[Camion], poubelles: List[Poubelle]) -> Dict[str, float]:
    total_volume = float(sum(p.volume_actuel_m3 for p in poubelles))
    total_capacite = float(sum(c.capacite_max_m3 for c in camions))

    if total_capacite <= 0:
        raise ValueError("Capacite totale de flotte invalide.")

    # La collecte totale est obligatoire: si volume > capacite instantanee,
    # on autorise des passages decharge (multi-trips) donc c'est faisable si camions existent.
    return {
        "total_volume_a_collecter_m3": round(total_volume, 4),
        "total_capacite_flotte_m3": round(total_capacite, 4),
        "ratio_charge_sur_capacite": round(total_volume / total_capacite, 4),
    }


def baseline_distance_individuelle_km(D: np.ndarray, n_poubelles: int, idx_decharge: int) -> float:
    # Baseline naive: depot -> poubelle -> decharge pour chaque poubelle
    return float(sum(D[0, i] + D[i, idx_decharge] for i in range(1, n_poubelles + 1)))


stats_contraintes = verifier_contraintes_globales(camions, poubelles)
baseline_km = baseline_distance_individuelle_km(D, len(poubelles), IDX_DECHARGE)
print(stats_contraintes)
print(f"Distance baseline naive (fin decharge): {baseline_km:.2f} km")

{'total_volume_a_collecter_m3': 22.3115, 'total_capacite_flotte_m3': 68.0, 'ratio_charge_sur_capacite': 0.3281}
Distance baseline naive (fin decharge): 3284.96 km


In [35]:
# 4) Generer les routes initiales avec Clarke & Wright

def demande_par_poubelle(poubelles: List[Poubelle]) -> Dict[str, float]:
    return {p.id: float(p.volume_actuel_m3) for p in poubelles}


def route_distance_km(route_bin_indices: List[int], D: np.ndarray) -> float:
    if not route_bin_indices:
        return 0.0
    seq = [0] + route_bin_indices + [IDX_DECHARGE]
    return float(sum(D[seq[i], seq[i + 1]] for i in range(len(seq) - 1)))


def partitionner_route_par_capacites(
    route: List[int],
    charges_idx: Dict[int, float],
    camions: List[Camion],
) -> List[List[int]]:
    if not route or len(camions) == 1:
        return [route]

    total_route = float(sum(charges_idx[idx] for idx in route))
    if total_route <= 0:
        return [route]

    total_caps = float(sum(c.capacite_max_m3 for c in camions))
    if total_caps <= 0:
        return [route]

    cibles = [total_route * (c.capacite_max_m3 / total_caps) for c in camions]

    segments: List[List[int]] = []
    cur_segment: List[int] = []
    cur_sum = 0.0
    target_idx = 0
    for node in route:
        cur_segment.append(node)
        cur_sum += charges_idx[node]

        if target_idx < len(cibles) - 1 and cur_sum >= cibles[target_idx]:
            segments.append(cur_segment)
            cur_segment = []
            cur_sum = 0.0
            target_idx += 1

    if cur_segment:
        segments.append(cur_segment)

    return [s for s in segments if s]


def clarke_wright_multi_camions(
    poubelles: List[Poubelle],
    camions: List[Camion],
    D: np.ndarray,
) -> List[Dict[str, Any]]:
    # Index poubelles dans D: 1..N
    n = len(poubelles)
    demandes = demande_par_poubelle(poubelles)
    id_by_idx = {i + 1: poubelles[i].id for i in range(n)}
    charges_idx = {i + 1: demandes[id_by_idx[i + 1]] for i in range(n)}

    routes = {i: [i] for i in range(1, n + 1)}
    route_charge = {i: charges_idx[i] for i in range(1, n + 1)}
    node_to_route = {i: i for i in range(1, n + 1)}

    capacite_ref = max(c.capacite_max_m3 for c in camions)

    savings = []
    for i in range(1, n + 1):
        for j in range(i + 1, n + 1):
            s = D[0, i] + D[0, j] - D[i, j]
            savings.append((s, i, j))
    savings.sort(reverse=True, key=lambda x: x[0])

    for _, i, j in savings:
        ri = node_to_route[i]
        rj = node_to_route[j]
        if ri == rj:
            continue

        route_i = routes[ri]
        route_j = routes[rj]

        i_is_end = i == route_i[0] or i == route_i[-1]
        j_is_end = j == route_j[0] or j == route_j[-1]
        if not (i_is_end and j_is_end):
            continue

        new_load = route_charge[ri] + route_charge[rj]
        if new_load > capacite_ref:
            continue

        if route_i[-1] == i and route_j[0] == j:
            merged = route_i + route_j
        elif route_i[0] == i and route_j[-1] == j:
            merged = route_j + route_i
        elif route_i[0] == i and route_j[0] == j:
            merged = list(reversed(route_i)) + route_j
        else:
            merged = route_i + list(reversed(route_j))

        routes[ri] = merged
        route_charge[ri] = new_load
        for node in route_j:
            node_to_route[node] = ri
        del routes[rj]
        del route_charge[rj]

    routes_list = list(routes.values())

    # Si Clarke-Wright fusionne trop en une seule tournee, on repartitionne
    # en segments proportionnels aux capacites pour utiliser toute la flotte.
    if len(routes_list) == 1 and len(camions) > 1:
        route_unique = routes_list[0]
        routes_list = partitionner_route_par_capacites(route_unique, charges_idx, camions)

    # Trier routes par charge decroissante pour mieux alimenter le seed multi-camions.
    routes_list.sort(key=lambda r: sum(charges_idx[idx] for idx in r), reverse=True)

    camions_tries = sorted(camions, key=lambda c: c.capacite_max_m3, reverse=True)
    plans = [{"camion_id": c.id, "route": [], "charge_m3": 0.0} for c in camions_tries]
    caps = {c.id: c.capacite_max_m3 for c in camions_tries}

    # Seed: distribuer d'abord un segment par camion (si possible)
    seed_count = min(len(plans), len(routes_list))
    for i in range(seed_count):
        r = routes_list[i]
        load_r = sum(charges_idx[idx] for idx in r)
        plans[i]["route"].extend(r)
        plans[i]["charge_m3"] += load_r

    # Puis insertion gloutonne pour les segments restants
    for r in routes_list[seed_count:]:
        load_r = sum(charges_idx[idx] for idx in r)
        best_plan = None
        best_score = float("inf")

        for plan in plans:
            cap = caps[plan["camion_id"]]
            futur_ratio = (plan["charge_m3"] + load_r) / max(1e-9, cap)
            if futur_ratio > 1.35:
                continue

            route_existante = plan["route"]
            extra = route_distance_km(route_existante + r, D) - route_distance_km(route_existante, D)
            score = extra + 8.0 * futur_ratio
            if score < best_score:
                best_score = score
                best_plan = plan

        if best_plan is None:
            best_plan = min(plans, key=lambda x: x["charge_m3"] / max(1e-9, caps[x["camion_id"]]))

        best_plan["route"].extend(r)
        best_plan["charge_m3"] += load_r

    # Retour dans l'ordre original des camions pour coherence de sortie
    order = {c.id: i for i, c in enumerate(camions)}
    plans.sort(key=lambda p: order[p["camion_id"]])

    return plans


plans_initiaux = clarke_wright_multi_camions(poubelles, camions, D)
for p in plans_initiaux:
    print(p["camion_id"], "stops:", len(p["route"]), "charge:", round(p["charge_m3"], 3))

CAM-01 stops: 113 charge: 14.402
CAM-02 stops: 67 charge: 7.909
CAM-03 stops: 0 charge: 0.0
CAM-04 stops: 0 charge: 0.0
CAM-05 stops: 0 charge: 0.0
CAM-06 stops: 0 charge: 0.0


In [36]:
# 5) Optimiser les tournees avec 2-opt et 3-opt

def two_opt(route: List[int], D: np.ndarray) -> List[int]:
    if len(route) < 4:
        return route
    best = route[:]
    improved = True
    while improved:
        improved = False
        best_dist = route_distance_km(best, D)
        for i in range(0, len(best) - 2):
            for k in range(i + 1, len(best) - 1):
                cand = best[:i] + list(reversed(best[i:k + 1])) + best[k + 1:]
                cand_dist = route_distance_km(cand, D)
                if cand_dist + 1e-9 < best_dist:
                    best = cand
                    best_dist = cand_dist
                    improved = True
    return best


def three_opt_simplifie(route: List[int], D: np.ndarray, max_iter: int = 40) -> List[int]:
    if len(route) < 6:
        return route
    best = route[:]
    best_dist = route_distance_km(best, D)
    n = len(route)
    iter_count = 0
    for i in range(1, n - 4):
        for j in range(i + 1, n - 2):
            for k in range(j + 1, n - 1):
                variants = [
                    best[:i] + list(reversed(best[i:j])) + best[j:k] + best[k:],
                    best[:i] + best[i:j] + list(reversed(best[j:k])) + best[k:],
                    best[:i] + list(reversed(best[i:j])) + list(reversed(best[j:k])) + best[k:],
                ]
                for cand in variants:
                    cand_dist = route_distance_km(cand, D)
                    if cand_dist + 1e-9 < best_dist:
                        best = cand
                        best_dist = cand_dist
                iter_count += 1
                if iter_count >= max_iter:
                    return best
    return best


def reequilibrer_tournees_par_faisabilite(
    plans: List[Dict[str, Any]],
    camions: List[Camion],
    max_arrets_par_camion: int = 70,
) -> List[Dict[str, Any]]:
    # Reequilibrage simple pour respecter des tournees exploitables
    # et activer la flotte lorsque certaines tournees sont sur-chargees en arrets.
    updated = copy.deepcopy(plans)
    id_to_plan = {p["camion_id"]: p for p in updated}

    idle = [p for p in updated if len(p["route"]) == 0]
    overloaded = sorted([p for p in updated if len(p["route"]) > max_arrets_par_camion], key=lambda x: len(x["route"]), reverse=True)

    for donor in overloaded:
        while len(donor["route"]) > max_arrets_par_camion and idle:
            receiver = idle.pop(0)
            surplus = len(donor["route"]) - max_arrets_par_camion
            chunk_size = max(15, surplus)
            chunk = donor["route"][-chunk_size:]
            donor["route"] = donor["route"][:-chunk_size]

            receiver["route"].extend(chunk)

    # Recalcul charge_m3 apres reequilibrage
    volume_map = {p.id: p.volume_actuel_m3 for p in poubelles}
    idx_to_id = {i + 1: poubelles[i].id for i in range(len(poubelles))}
    for plan in updated:
        plan["charge_m3"] = float(sum(volume_map[idx_to_id[idx]] for idx in plan["route"]))

    return updated


plans_optimises = copy.deepcopy(plans_initiaux)
for plan in plans_optimises:
    r = plan["route"]
    r2 = two_opt(r, D)
    r3 = three_opt_simplifie(r2, D)
    plan["route"] = r3

plans_optimises = reequilibrer_tournees_par_faisabilite(plans_optimises, camions, max_arrets_par_camion=70)

print("Optimisation locale terminee (2-opt/3-opt simplifie + reequilibrage faisabilite).")
for p in plans_optimises:
    print(p["camion_id"], "stops:", len(p["route"]), "charge:", round(p["charge_m3"], 3))

Optimisation locale terminee (2-opt/3-opt simplifie + reequilibrage faisabilite).
CAM-01 stops: 70 charge: 7.122
CAM-02 stops: 67 charge: 7.909
CAM-03 stops: 43 charge: 7.28
CAM-04 stops: 0 charge: 0.0
CAM-05 stops: 0 charge: 0.0
CAM-06 stops: 0 charge: 0.0


In [37]:
# 6) Arbitrer la saturation de capacite : scenario A vs scenario B

def volume_route(route: List[int], poubelles: List[Poubelle]) -> float:
    return float(sum(poubelles[idx - 1].volume_actuel_m3 for idx in route))


def arbitre_saturation(
    camion_id: str,
    route_restante: List[int],
    plan_by_camion: Dict[str, Dict[str, Any]],
    camions: List[Camion],
    poubelles: List[Poubelle],
    D: np.ndarray,
) -> Dict[str, Any]:
    if not route_restante:
        return {"decision": "aucune", "camion_relais": None, "cout_delta_km": 0.0}

    cam_map = {c.id: c for c in camions}
    active = plan_by_camion[camion_id]
    cur_load = active.get("charge_courante_m3", 0.0)
    cap = cam_map[camion_id].capacite_max_m3

    first_stop = route_restante[0]

    # Scenario B: vidange a la decharge puis retour sur la tournee
    cout_B = float(D[first_stop, IDX_DECHARGE] + D[IDX_DECHARGE, first_stop])

    # Scenario A: transfert vers camion voisin ayant de la capacite
    best_A = {"decision": "scenario_B_retour_tournee", "camion_relais": None, "cout_delta_km": cout_B}

    for other_id, other_plan in plan_by_camion.items():
        if other_id == camion_id:
            continue

        other_cap = cam_map[other_id].capacite_max_m3
        other_cur = other_plan.get("charge_courante_m3", 0.0)
        vol_rest = volume_route(route_restante, poubelles)
        if other_cur + vol_rest > other_cap:
            continue

        if other_plan["route"]:
            pos_other = other_plan["route"][0]
            cout_transfert = float(D[pos_other, first_stop])
        else:
            cout_transfert = float(D[0, first_stop])

        cout_A = float(D[first_stop, IDX_DECHARGE] + cout_transfert)
        if cout_A + 1e-9 < best_A["cout_delta_km"]:
            best_A = {
                "decision": "scenario_A_transfert_voisin",
                "camion_relais": other_id,
                "cout_delta_km": cout_A,
            }

    if cur_load >= cap:
        return best_A
    return {"decision": "aucune", "camion_relais": None, "cout_delta_km": 0.0}


plan_by_camion = {p["camion_id"]: {**p, "charge_courante_m3": 0.0} for p in plans_optimises}
print("Arbitrage capacite pret.")

Arbitrage capacite pret.


In [38]:
# 7) Simuler les durees et verifier la faisabilite horaire

def estimer_temps_et_distance_route(route: List[int], D: np.ndarray, passages_decharge: int) -> Tuple[float, float]:
    dist_km = route_distance_km(route, D)
    temps_trajet_min = (dist_km / VITESSE_URBAINE_KMH) * 60.0
    temps_levees_min = len(route) * TEMPS_LEVEE_MIN
    temps_decharge_min = passages_decharge * TEMPS_VIDANGE_MIN
    total_min = temps_trajet_min + temps_levees_min + temps_decharge_min
    return float(dist_km), float(total_min)


def verifier_faisabilite_journee(plan_by_camion: Dict[str, Dict[str, Any]], D: np.ndarray) -> Dict[str, Any]:
    rapport = {}
    limite = HEURE_RETOUR_LIMITE_MIN - HEURE_DEPART_MIN
    for camion_id, plan in plan_by_camion.items():
        passages = int(plan.get("passages_decharge", 0))
        dist_km, duree = estimer_temps_et_distance_route(plan["route"], D, passages)
        rapport[camion_id] = {
            "distance_totale_km": round(dist_km, 2),
            "duree_estimee_min": int(round(duree)),
            "faisable_avant_17h": bool(duree <= limite),
        }
    return rapport


for k in plan_by_camion:
    plan_by_camion[k]["passages_decharge"] = 0

faisabilite = verifier_faisabilite_journee(plan_by_camion, D)
print(pd.DataFrame(faisabilite).T)

       distance_totale_km duree_estimee_min faisable_avant_17h
CAM-01              60.16               354               True
CAM-02              62.27               350               True
CAM-03              30.31               202               True
CAM-04                0.0                 0               True
CAM-05                0.0                 0               True
CAM-06                0.0                 0               True


In [39]:
# 8) Gerer le recalcul dynamique en cas d'anomalie

def replanifier_apres_anomalie(
    plan_by_camion: Dict[str, Dict[str, Any]],
    camions: List[Camion],
    poubelles: List[Poubelle],
    anomalie: Dict[str, Any],
    D: np.ndarray,
) -> Dict[str, Dict[str, Any]]:
    updated = copy.deepcopy(plan_by_camion)

    type_anomalie = anomalie.get("type")
    camion_id = anomalie.get("camion_id")
    poubelle_id = anomalie.get("poubelle_id")

    id_to_idx = {p.id: i + 1 for i, p in enumerate(poubelles)}

    if type_anomalie == "poubelle_inaccessible" and poubelle_id in id_to_idx:
        idx = id_to_idx[poubelle_id]
        # Retirer l'arret pour le camion courant puis reassigner au camion le plus proche
        for cid, pl in updated.items():
            if idx in pl["route"]:
                pl["route"] = [x for x in pl["route"] if x != idx]
                break

        best_cid = min(updated.keys(), key=lambda cid: D[0, idx] if not updated[cid]["route"] else D[updated[cid]["route"][0], idx])
        updated[best_cid]["route"].append(idx)

    elif type_anomalie == "camion_en_panne" and camion_id in updated:
        restants = updated[camion_id]["route"][:]
        updated[camion_id]["route"] = []
        for idx in restants:
            best_cid = min(
                [cid for cid in updated.keys() if cid != camion_id],
                key=lambda cid: D[0, idx] if not updated[cid]["route"] else D[updated[cid]["route"][-1], idx],
            )
            updated[best_cid]["route"].append(idx)

    elif type_anomalie == "route_bloquee":
        # Re-optimisation locale simple de toutes les routes restantes
        for cid in updated:
            updated[cid]["route"] = two_opt(updated[cid]["route"], D)

    return updated


anomalie_demo = {"type": "route_bloquee", "camion_id": "CAM-01"}
plan_recalcule = replanifier_apres_anomalie(plan_by_camion, camions, poubelles, anomalie_demo, D)
print("Recalcul dynamique pret.")

Recalcul dynamique pret.


In [40]:
# 9) Exporter le planning de tournee au format JSON

def formatter_sequence(
    route: List[int],
    poubelles: List[Poubelle],
    decharge: Point,
    plan_by_camion: Dict[str, Dict[str, Any]],
    camion_id: str,
    camions: List[Camion],
    D: np.ndarray,
) -> List[Dict[str, Any]]:
    seq = []
    cam_map = {c.id: c for c in camions}
    charge = 0.0
    ordre = 1
    remaining = route[:]

    while remaining:
        idx = remaining.pop(0)
        pb = poubelles[idx - 1]
        charge += pb.volume_actuel_m3

        seq.append(
            {
                "ordre": ordre,
                "type": "collecte",
                "poubelle_id": pb.id,
                "position": {"lat": pb.position.lat, "lng": pb.position.lng},
                "volume_a_collecter_m3": round(pb.volume_actuel_m3, 4),
                "taux_remplissage_pct": round(pb.taux_remplissage_pct, 2),
                "statut": "en_attente",
            }
        )
        ordre += 1

        cap = cam_map[camion_id].capacite_max_m3
        if charge >= cap:
            decision = arbitre_saturation(camion_id, remaining, plan_by_camion, camions, poubelles, D)
            seq.append(
                {
                    "ordre": ordre,
                    "type": "vidange_decharge",
                    "position": {"lat": decharge.lat, "lng": decharge.lng},
                    "raison": "capacite_atteinte",
                    "decision": decision["decision"],
                    "camion_relais": decision["camion_relais"],
                }
            )
            ordre += 1
            charge = 0.0

    # Fin obligatoire de tournee a la decharge
    seq.append(
        {
            "ordre": ordre,
            "type": "fin_tournee_decharge",
            "position": {"lat": decharge.lat, "lng": decharge.lng},
            "raison": "fin_tournee",
            "decision": "retour_decharge_obligatoire",
            "camion_relais": None,
        }
    )

    return seq


def exporter_planning_json(
    date_plan: str,
    heure_generation: str,
    plan_by_camion: Dict[str, Dict[str, Any]],
    poubelles: List[Poubelle],
    camions: List[Camion],
    decharge: Point,
    D: np.ndarray,
    chauffeur_by_camion: Dict[str, str] | None = None,
) -> Dict[str, Any]:
    if chauffeur_by_camion is None:
        chauffeur_by_camion = {c.id: f"CHAUFFEUR_{c.id}" for c in camions}

    tournees = []
    for camion in camions:
        plan = plan_by_camion[camion.id]
        sequence = formatter_sequence(plan["route"], poubelles, decharge, plan_by_camion, camion.id, camions, D)

        passages_decharge = sum(1 for s in sequence if s["type"] in ("vidange_decharge", "fin_tournee_decharge"))
        dist_km, duree_min = estimer_temps_et_distance_route(plan["route"], D, passages_decharge)

        tournees.append(
            {
                "camion_id": camion.id,
                "chauffeur": chauffeur_by_camion.get(camion.id, "N/A"),
                "capacite_chargee_m3": round(volume_route(plan["route"], poubelles), 3),
                "distance_totale_km": round(dist_km, 2),
                "duree_estimee_min": int(round(duree_min)),
                "nombre_arrets": len([s for s in sequence if s["type"] == "collecte"]),
                "passages_decharge": passages_decharge,
                "sequence": sequence,
            }
        )

    return {
        "date": date_plan,
        "heure_generation": heure_generation,
        "tournees": tournees,
    }

print("Export JSON pret.")

Export JSON pret.


In [41]:
# 10) Calculer les statistiques globales et les gains operationnels

def construire_statistiques_globales(
    output: Dict[str, Any],
    total_poubelles: int,
    baseline_km: float,
) -> Dict[str, Any]:
    total_collectees = int(sum(t["nombre_arrets"] for t in output["tournees"]))
    total_distance = float(sum(t["distance_totale_km"] for t in output["tournees"]))
    total_passages = int(sum(t["passages_decharge"] for t in output["tournees"]))

    economies = 0.0
    if baseline_km > 0:
        economies = max(0.0, (baseline_km - total_distance) / baseline_km * 100)

    return {
        "total_poubelles": total_poubelles,
        "total_poubelles_collectees": total_collectees,
        "taux_couverture_pct": round((total_collectees / total_poubelles) * 100, 2) if total_poubelles else 0.0,
        "total_distance_km": round(total_distance, 2),
        "total_passages_decharge": total_passages,
        "economies_vs_route_standard_pct": round(economies, 2),
    }


def charger_chauffeurs_depuis_trucks_csv(path_csv: str = "trucks.csv") -> Dict[str, str]:
    p = Path(path_csv)
    if not p.exists():
        return {c.id: f"CHAUFFEUR_{c.id}" for c in camions}
    df = pd.read_csv(p)
    if not {"truck_id", "driver"}.issubset(df.columns):
        return {c.id: f"CHAUFFEUR_{c.id}" for c in camions}
    mapping = {str(r["truck_id"]): str(r["driver"]) for _, r in df.iterrows()}
    for c in camions:
        mapping.setdefault(c.id, f"CHAUFFEUR_{c.id}")
    return mapping


def evaluer_qualite_historique(output: Dict[str, Any], date_plan: str) -> Dict[str, Any]:
    rapport = {
        "historique_disponible": False,
        "distance_reference_meme_date_km": None,
        "distance_moyenne_historique_km": None,
        "ecart_vs_reference_pct": None,
        "datasets_utiles": False,
    }

    p_opt = Path("optimal_routes.json")
    if not p_opt.exists():
        return rapport

    with open(p_opt, "r", encoding="utf-8") as f:
        hist = json.load(f)

    if not isinstance(hist, list) or not hist:
        return rapport

    distances = [float(day.get("total_fleet_distance_km", 0.0)) for day in hist if day.get("total_fleet_distance_km") is not None]
    if distances:
        rapport["distance_moyenne_historique_km"] = round(float(np.mean(distances)), 2)

    ref_day = next((day for day in hist if str(day.get("date")) == str(date_plan)), None)
    if ref_day is not None:
        ref_dist = float(ref_day.get("total_fleet_distance_km", 0.0))
        cur_dist = float(output["statistiques_globales"]["total_distance_km"])
        rapport["historique_disponible"] = True
        rapport["distance_reference_meme_date_km"] = round(ref_dist, 2)
        if ref_dist > 0:
            rapport["ecart_vs_reference_pct"] = round((cur_dist - ref_dist) / ref_dist * 100.0, 2)

    p_pairs = Path("training_pairs.json")
    if p_pairs.exists():
        with open(p_pairs, "r", encoding="utf-8") as f:
            pairs = json.load(f)
        if isinstance(pairs, list) and len(pairs) > 0:
            rapport["datasets_utiles"] = True

    return rapport


# Pipeline end-to-end
chauffeur_map = charger_chauffeurs_depuis_trucks_csv("trucks.csv")

output = exporter_planning_json(
    date_plan=date_plan,
    heure_generation=heure_generation,
    plan_by_camion=plan_recalcule,
    poubelles=poubelles,
    camions=camions,
    decharge=decharge,
    D=D,
    chauffeur_by_camion=chauffeur_map,
)
output["statistiques_globales"] = construire_statistiques_globales(output, len(poubelles), baseline_km)

# Verrou contrainte absolue: collecte totale obligatoire
if output["statistiques_globales"]["total_poubelles_collectees"] != output["statistiques_globales"]["total_poubelles"]:
    raise RuntimeError("Contrainte violee: toutes les poubelles doivent etre collectees.")

# Evaluation de l'apport des datasets historiques
rapport_historique = evaluer_qualite_historique(output, date_plan)
output["evaluation_historique"] = rapport_historique

# Sauvegarde du plan JSON
with open("plan_tournee_optimise.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("Plan genere et sauvegarde: plan_tournee_optimise.json")
print(json.dumps(output["statistiques_globales"], ensure_ascii=False, indent=2))
print("\nEvaluation historique:")
print(json.dumps(output["evaluation_historique"], ensure_ascii=False, indent=2))

# TODO - 3 cas metier a injecter plus tard
cas_metier_a_ajouter = {
    "cas_1": "A definir (ex: zone dense centre-ville)",
    "cas_2": "A definir (ex: saturation multi-camions simultanee)",
    "cas_3": "A definir (ex: panne camion + route barree)",
}
print("\nPlaceholders cas metier:")
print(json.dumps(cas_metier_a_ajouter, ensure_ascii=False, indent=2))

Plan genere et sauvegarde: plan_tournee_optimise.json
{
  "total_poubelles": 180,
  "total_poubelles_collectees": 180,
  "taux_couverture_pct": 100.0,
  "total_distance_km": 151.23,
  "total_passages_decharge": 6,
  "economies_vs_route_standard_pct": 95.4
}

Evaluation historique:
{
  "historique_disponible": true,
  "distance_reference_meme_date_km": 523.98,
  "distance_moyenne_historique_km": 512.13,
  "ecart_vs_reference_pct": -71.14,
  "datasets_utiles": true
}

Placeholders cas metier:
{
  "cas_1": "A definir (ex: zone dense centre-ville)",
  "cas_2": "A definir (ex: saturation multi-camions simultanee)",
  "cas_3": "A definir (ex: panne camion + route barree)"
}
